In [1]:
import pandas as pd
model_data = pd.read_csv('../data/processed/onion_model_ready.csv', parse_dates=['Arrival_Date'])
model_data = model_data.sort_values(['Market', 'Arrival_Date']).reset_index(drop=True)

#### **deviation from own market's recent history**

In [2]:
model_data['Own_Rolling_Median_30'] = (
    model_data
    .groupby('Market')['Modal_Price']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=10).median())
)
model_data['Deviation_From_Own_History'] = (
    model_data['Modal_Price'] / model_data['Own_Rolling_Median_30']
)

In [6]:
model_data['Same_Day_Regional_Avg'] = (
    model_data
    .groupby('Arrival_Date')['Modal_Price']
    .transform('mean')
)
model_data['Deviation_From_Regional'] = (
    model_data['Modal_Price'] / model_data['Same_Day_Regional_Avg']
)

In [7]:
model_data['Potential_Unfair_Price'] = (
    (model_data['Deviation_From_Own_History'] < 0.7) &
    (model_data['Deviation_From_Regional'] < 0.7)
)

print(model_data['Potential_Unfair_Price'].sum(),"flagged rows out of", len(model_data))

168 flagged rows out of 33895


In [9]:
model_data[model_data['Potential_Unfair_Price']][['Market','Arrival_Date','Modal_Price','Own_Rolling_Median_30','Same_Day_Regional_Avg','Deviation_From_Own_History','Deviation_From_Regional']].head(10)

,Market,Arrival_Date,Modal_Price,Own_Rolling_Median_30,Same_Day_Regional_Avg,Deviation_From_Own_History,Deviation_From_Regional
152,Chandvad,2004-05-14,170.0,275.0,277.500000,0.618182,0.612613
278,Chandvad,2005-06-09,200.0,295.0,293.400000,0.677966,0.681663
620,Chandvad,2009-11-25,950.0,1400.0,1375.166667,0.678571,0.690825
623,Chandvad,2009-12-02,951.0,1400.0,1525.285714,0.679286,0.623490
635,Chandvad,2009-12-21,850.0,1300.0,1428.571429,0.653846,0.595000
640,Chandvad,2009-12-28,800.0,1300.0,1412.625000,0.615385,0.566322
832,Chandvad,2010-12-14,850.0,1400.0,2183.416667,0.607143,0.389298
998,Chandvad,2011-11-16,400.0,865.0,686.642857,0.462428,0.582544
1006,Chandvad,2011-12-02,400.0,800.0,599.857143,0.500000,0.666825
1008,Chandvad,2011-12-07,400.0,800.0,582.785714,0.500000,0.686359


In [10]:
model_data.to_csv('../data/processed/onion_with_anomalies.csv', index=False)